<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3%2BResUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [ ]:
import numpy as np
import tensorflow as tf

# Define the ESA class map
class_map = {
    10: 0, # Tree cover
    20: 1, # Shrubland
    30: 2, # Grassland
    40: 3, # Cropland
    60: 3, # Bare / Sparse vegetation (Combined with Cropland)
    50: 4, # Built-up
    80: 5, # Permanent water bodies
    90: 5, # Herbaceous wetland (Combined with Water Bodies)
}


lut = np.full(101, -1, dtype=np.int32)
for old_id, new_id in class_map.items():
    lut[old_id] = new_id

# Apply mapping
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]


if Y_train_ready.ndim == 3:
    Y_train_ready = np.expand_dims(Y_train_ready, axis=-1)
    Y_test_ready = np.expand_dims(Y_test_ready, axis=-1)

print(f"Unique IDs in merged train mask: {np.unique(Y_train_ready)}")
print(f"Final Y_train_ready shape: {Y_train_ready.shape}")

Unique IDs in merged train mask: [0 1 2 3 4 5]
Final Y_train_ready shape: (639, 256, 256, 1)


In [ ]:
# Converting for RAM efficiency

# Imagery must be float32 for the model weights
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# Labels must be int32 for Sparse Categorical Crossentropy
Y_train_ready = Y_train_ready.astype('int32')
Y_test_ready = Y_test_ready.astype('int32')

In [ ]:
def residual_block(x, filters, dropout_rate=0.3):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dropout(dropout_rate)(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_inception_res_unet(input_shape=(256, 256, 7), num_classes=6):
    inputs = layers.Input(input_shape)

    # THE 7-BAND ADAPTER
    adapter = layers.Conv2D(3, (1, 1), padding='same', name='band_adapter')(inputs)

    # INCEPTIONV3 ENCODER
    inception_base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights='imagenet',
        input_shape=(256, 256, 3)
    )

    s1 = inception_base.get_layer(index=11).output    # ~125x125 (Initial Conv/Act)
    s2 = inception_base.get_layer(index=17).output    # ~61x61
    s3 = inception_base.get_layer("mixed2").output    # 28x28
    bridge = inception_base.get_layer("mixed7").output # 12x12

    encoder_model = models.Model(inputs=inception_base.input, outputs=[s1, s2, s3, bridge])
    enc_s1, enc_s2, enc_s3, enc_bridge = encoder_model(adapter)

    # RESIDUAL DECODER
    u1 = layers.UpSampling2D((2, 2))(enc_bridge)
    u1 = layers.Resizing(enc_s3.shape[1], enc_s3.shape[2])(u1)
    u1 = layers.concatenate([u1, enc_s3])
    d1 = residual_block(u1, 128)

    u2 = layers.UpSampling2D((2, 2))(d1)
    u2 = layers.Resizing(enc_s2.shape[1], enc_s2.shape[2])(u2)
    u2 = layers.concatenate([u2, enc_s2])
    d2 = residual_block(u2, 64)

    u3 = layers.UpSampling2D((2, 2))(d2)
    u3 = layers.Resizing(enc_s1.shape[1], enc_s1.shape[2])(u3)
    u3 = layers.concatenate([u3, enc_s1])
    d3 = residual_block(u3, 32)

    u_final = layers.UpSampling2D((2, 2))(d3)
    u_final = layers.Resizing(256, 256)(u_final)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(u_final)

    return models.Model(inputs, outputs, name="InceptionV3_ResUNet")

# Clear the Keras backend
tf.keras.backend.clear_session()

model = build_inception_res_unet()
print("Model built successfully!")

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Model built successfully!


In [ ]:
import numpy as np
from sklearn.utils import class_weight

# Calculate Class Weights to handle imbalance
y_flat = Y_train_ready.flatten()
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_flat),
    y=y_flat
)
class_weights_dict = dict(enumerate(weights))
print("Computed Class Weights", weights)

Computed Class Weights [1.11912802 1.041065   0.75546067 0.97782913 1.32307165 0.95812405]


In [ ]:
# Create a 2D weight map for each image
def create_sample_weights(masks, weights_dict):
    sample_weights = np.ones(masks.shape, dtype='float32')
    for class_id, weight in weights_dict.items():
        sample_weights[masks == class_id] = weight
    return sample_weights

# Generate weights for train and test
train_sample_weights = create_sample_weights(Y_train_ready, class_weights_dict)

In [ ]:
from sklearn.metrics import f1_score

class TableLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val_integers = val_data

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<10} | {'Val Loss':<10} | {'Val Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 65)

    def on_epoch_end(self, epoch, logs=None):
        val_logits = self.model.predict(self.X_val, verbose=0, batch_size=8)
        val_preds = np.argmax(val_logits, axis=-1).flatten()
        val_true = self.y_val_integers.flatten()

        val_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<10.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<10.4f}")


In [ ]:
import time
import psutil

process = psutil.Process()
start_time = time.time()

# Compile the Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

table_logger = TableLogger(val_data=(X_test, Y_test_ready))

# Start training
history = model.fit(
    X_train, Y_train_ready,
    validation_data=(X_test, Y_test_ready),
    epochs=25,
    batch_size=16,
    sample_weight=train_sample_weights,
    callbacks=[table_logger, reduce_lr],
    verbose=0
)

end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} s")
print(f"System RAM Usage: {process.memory_info().rss / 1024**2:.2f} MB")


Epoch  | Train Loss | Val Loss   | Val Acc  | F1 (Macro)
-----------------------------------------------------------------
1      | 1.3482     | 1.4286     | 0.5162   | 0.4200    
2      | 0.9223     | 1.1211     | 0.6115   | 0.5650    
3      | 0.8092     | 0.9367     | 0.6550   | 0.6243    
4      | 0.7452     | 0.8698     | 0.6672   | 0.6406    
5      | 0.7064     | 0.8427     | 0.6743   | 0.6444    
6      | 0.6922     | 0.7384     | 0.7201   | 0.7024    
7      | 0.6527     | 0.7503     | 0.7108   | 0.6848    
8      | 0.6494     | 0.8026     | 0.6847   | 0.6698    
9      | 0.6262     | 0.6906     | 0.7306   | 0.7077    
10     | 0.6043     | 0.6612     | 0.7426   | 0.7240    
11     | 0.6020     | 0.6690     | 0.7386   | 0.7231    
12     | 0.5827     | 0.6579     | 0.7412   | 0.7247    
13     | 0.5814     | 0.6580     | 0.7438   | 0.7240    
14     | 0.5593     | 0.6601     | 0.7385   | 0.7183    
15     | 0.5490     | 0.6501     | 0.7420   | 0.7223    
16     | 0.5460     |

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import gc

print("Generating predictions...")

# Predict and immediately convert to indices to save RAM
train_preds = np.argmax(model.predict(X_train, batch_size=8, verbose=1), axis=-1).flatten()
test_preds = np.argmax(model.predict(X_test, batch_size=8, verbose=1), axis=-1).flatten()

y_train_flat = Y_train_ready.flatten()
y_test_flat = Y_test_ready.flatten()

gc.collect()

# Calculate Overall Metrics
train_acc = accuracy_score(y_train_flat, train_preds)
test_acc = accuracy_score(y_test_flat, test_preds)
test_f1_macro = f1_score(y_test_flat, test_preds, average="macro")

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1_macro:.4f}")

# Detailed Classification Report
merged_names = [
    "Tree cover",
    "Shrubland",
    "Grassland",
    "Cropland/vegetation",
    "Built-up",
    "Permanent water bodies"
]

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test_flat, test_preds, target_names=merged_names))

Generating predictions...
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 135ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step
Train Accuracy: 0.8178
Test Accuracy:  0.7433
Test Macro F1:  0.7259

--- CLASSIFICATION REPORT ---
                        precision    recall  f1-score   support

            Tree cover       0.62      0.61      0.61    829607
             Shrubland       0.61      0.59      0.60   1016613
             Grassland       0.58      0.52      0.55   1772697
   Cropland/vegetation       0.77      0.81      0.79   1822707
              Built-up       0.77      0.88      0.82   1090768
Permanent water bodies       0.99      0.98      0.99   1659608

              accuracy                           0.74   8192000
             macro avg       0.72      0.73      0.73   8192000
          weighted avg       0.74      0.74      0.74   8192000



In [ ]:
# Patch conversion
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score, cohen_kappa_score, jaccard_score

def pixels_to_patch_label(y_array):
    """
    Converts (N, 256, 256, 1) or (N, 256, 256) pixel masks
    into (N,) patch labels using the majority class.
    """
    patch_labels = []
    for i in range(y_array.shape[0]):
        # Flatten the 256x256 patch into a 1D list of pixels
        pixels = y_array[i].flatten()

        # Find the most frequent class (the mode)
        counts = np.bincount(pixels, minlength=6)
        majority_class = np.argmax(counts)
        patch_labels.append(majority_class)

    return np.array(patch_labels)

# Convert Ground Truth
y_test_patch_true = pixels_to_patch_label(Y_test_ready)

# Convert Your Model's Predictions
test_preds_spatial = test_preds.reshape(-1, 256, 256)
y_test_patch_pred = pixels_to_patch_label(test_preds_spatial)

# Calculate Patch-Level Accuracy
patch_acc      = accuracy_score(y_test_patch_true, y_test_patch_pred)
patch_f1_macro = f1_score(y_test_patch_true, y_test_patch_pred, average='macro')
patch_kappa    = cohen_kappa_score(y_test_patch_true, y_test_patch_pred)
patch_iou      = jaccard_score(y_test_patch_true, y_test_patch_pred, average='macro')

print(f"Test Accuracy (Patch): {patch_acc:.4f}")
print(f"Test Macro F1 (Patch): {patch_f1_macro:.4f}")
print(f"Cohen's Kappa: {patch_kappa:.4f}")
print(f"Mean IoU:      {patch_iou:.4f}")

print("\n--- PATCH-LEVEL CLASSIFICATION REPORT ---")
print(classification_report(y_test_patch_true, y_test_patch_pred, target_names=merged_names))

Test Accuracy (Patch): 0.8960
Test Macro F1 (Patch): 0.8966
Cohen's Kappa: 0.8711
Mean IoU:      0.8255

--- PATCH-LEVEL CLASSIFICATION REPORT ---
                        precision    recall  f1-score   support

            Tree cover       1.00      1.00      1.00         6
             Shrubland       0.77      0.77      0.77        13
             Grassland       0.90      0.68      0.78        28
   Cropland/vegetation       0.88      0.97      0.92        29
              Built-up       0.85      1.00      0.92        22
Permanent water bodies       1.00      1.00      1.00        27

              accuracy                           0.90       125
             macro avg       0.90      0.90      0.90       125
          weighted avg       0.90      0.90      0.89       125

